In [1]:
import pandas as pd
import numpy as np
from decimal import Decimal, ROUND_HALF_UP
from xgboost import XGBClassifier
import xgboost as xgb
from lightgbm import LGBMClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [2]:
case_df = pd.read_csv('s3://ads508-s3/time_series_covid19_confirmed_US.csv')
case_df.head()

PermissionError: Forbidden

In [ ]:
death_df = pd.read_csv('s3://ads508-s3/time_series_covid19_deaths_US.csv')
death_df.head()

In [ ]:
case_df.columns

In [ ]:
print(case_df.shape)
print(death_df.shape)

In [ ]:
case_df = case_df.drop(
    columns=['UID', 'iso2', 'iso3', 'code3', 'FIPS', 'Country_Region', 'Combined_Key'],
    errors='ignore'
)

# Drop rows where 'Admin2' is missing
case_df = case_df.dropna(subset=['Admin2'])

# Reshape the dataframe from wide to long
case_df = pd.melt(
    case_df,
    id_vars=['Admin2', 'Province_State', 'Lat', 'Long_'],
    var_name='Date',
    value_name='Cases'  # Use different name to avoid conflict
)

# Convert 'Date' column with explicit format
case_df['Date'] = pd.to_datetime(case_df['Date'], format='%m/%d/%y')

# Optimize memory usage
case_df['Cases'] = pd.to_numeric(case_df['Cases'], downcast='integer')
case_df['Lat'] = np.round(case_df['Lat'], 3)
case_df['Long_'] = np.round(case_df['Long_'], 3)
case_df['Admin2'] = case_df['Admin2'].astype('category')
case_df['Province_State'] = case_df['Province_State'].astype('category')

# Resample weekly death data
case_df.set_index('Date', inplace=True)
case_resample = case_df.groupby(
    ['Admin2', 'Lat', 'Long_', 'Province_State'],
    observed=False
).resample('W')['Cases'].mean().reset_index()

# Final cleanup
case_resample.reset_index(drop=True, inplace=True)

In [ ]:
death_df = death_df.drop(
    columns=['UID', 'iso2', 'iso3', 'code3', 'FIPS', 'Country_Region', 'Combined_Key'],
    errors='ignore'
)

# Drop rows where 'Admin2' is missing
death_df = death_df.dropna(subset=['Admin2'])

# Reshape the DataFrame from wide to long
death_df = pd.melt(
    death_df,
    id_vars=['Admin2', 'Province_State', 'Lat', 'Long_', 'Population'],
    var_name='Date',
    value_name='Deaths'
)

# Convert 'Date' column with explicit format
death_df['Date'] = pd.to_datetime(death_df['Date'], format='%m/%d/%y')

# Optimize memory usage
death_df['Deaths'] = pd.to_numeric(death_df['Deaths'], downcast='integer')
death_df['Population'] = pd.to_numeric(death_df['Population'], downcast='integer')
death_df['Lat'] = np.round(death_df['Lat'], 3)
death_df['Long_'] = np.round(death_df['Long_'], 3)
death_df['Admin2'] = death_df['Admin2'].astype('category')
death_df['Province_State'] = death_df['Province_State'].astype('category')

# Resample weekly confirmed data
death_df.set_index('Date', inplace=True)
death_resample = death_df.groupby(
    ['Admin2', 'Lat', 'Long_', 'Province_State', 'Population'],
    observed=False
).resample('W')['Deaths'].mean().reset_index()

# Final cleanup
death_resample.reset_index(drop=True, inplace=True)

In [ ]:
death_staging = death_resample[['Date', 'Admin2', 'Lat', 'Long_', 'Province_State', 'Deaths', 'Population']]

# Merge on full key set to avoid many-to-many explosion
base_df = pd.merge(
    case_resample,
    death_staging,
    on=['Date', 'Admin2', 'Lat', 'Long_', 'Province_State'],
    how='left'
)

base_df.sort_values(['Admin2', 'Date'], inplace = True)
base_df.reset_index(inplace=True, drop = True)
base_df.head()

In [ ]:
base_df.shape

In [ ]:
print(base_df['Date'].min())
print(base_df['Date'].max())

In [ ]:
"""The following are the top 10 states with the most covid deaths between 01-26-2020 and 03-12-2023... The numbers are not adding up properly need to look more into this"""
(
    base_df
    .groupby('Province_State', observed=False)['Deaths']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

## Feature Engineering

In [ ]:
#Avg Case Stats by State

base_df['Avg State Cases'] = base_df.groupby(['Date', 'Province_State'])['Cases'].transform('mean')
base_df['Avg State Deaths'] = base_df.groupby(['Date', 'Province_State'])['Deaths'].transform('mean')

In [ ]:
windows = {
    '1_month': 4,
    '2_month': 8,
    '3_month': 12
}

# Calculate rolling averages for each city
for window_name, window_size in windows.items():
    base_df[f'cases_{window_name}_avg'] = (
        base_df.groupby('Admin2')['Cases']
        .transform(lambda x: x.rolling(window=window_size, min_periods=1).mean())
    )
    base_df[f'deaths_{window_name}_avg'] = (
        base_df.groupby('Admin2')['Deaths']
        .transform(lambda x: x.rolling(window=window_size, min_periods=1).mean())
    )


In [ ]:
base_df.sort_values(['Admin2', 'Date'], inplace = True)
base_df.reset_index(inplace=True, drop = True)

for lag in [1, 2, 3]:
    base_df[f'lag_{lag}_week_cases'] = (
        base_df
        .groupby('Admin2')['Cases']
        .shift(lag)
    )
    base_df[f'lag_{lag}_week_deaths'] = (
        base_df
        .groupby('Admin2')['Deaths']
        .shift(lag)
    )


In [ ]:
numerical_cols = base_df.select_dtypes(include='number').columns.drop(['Population', 'Lat', 'Long_'])

# Create per-capita versions of all stats
base_df[[f"{col}_per_capita" for col in numerical_cols]] = (
    base_df[numerical_cols].div(base_df['Population'], axis=0)
)


In [ ]:
base_df['weekofyear'] = base_df['Date'].dt.isocalendar().week
base_df['month'] = base_df['Date'].dt.month
base_df['year'] = base_df['Date'].dt.year

In [ ]:
# Create shifted per capita values for comparison
base_df['nextweek_case_rate'] = base_df.groupby('Admin2')['Cases_per_capita'].shift(-1)

# Create binary target indicator 
#base_df['is_hotspot'] = (base_df['nextweek_case_rate'] >= 0.1).astype(int)
base_df['is_hotspot'] = ((base_df['nextweek_case_rate'] > base_df['Cases_per_capita']) & (base_df['nextweek_case_rate'] >= 0.1)).astype(int)

base_df = base_df.drop(columns=['nextweek_case_rate'])

In [ ]:
base_df[10000:10100]

In [ ]:
base_df['is_hotspot'].value_counts(normalize=True)

## Model Implementation and Testing

In [ ]:
df = base_df.dropna()
df = df.replace([np.inf, -np.inf], np.nan).dropna()

#Create monthly strata 
df['time_period'] = df['Date'].dt.to_period('Q')

train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df[['Admin2', 'time_period']],  # Combine city and time strata
    random_state=42
)

df = df.drop(columns=['time_period'])

feature_cols = []

for col in df.columns:
    
    if col not in ['Admin2', 'Province_State', 'Date', 'is_hotspot']:
        feature_cols.append(col)

X_train = train_df[feature_cols]
y_train = train_df['is_hotspot']

X_test = test_df[feature_cols]
y_test = test_df['is_hotspot']


In [ ]:
models = {
    'XGBoost': XGBClassifier(
        tree_method='hist',          # CPU-optimized tree method
        use_label_encoder=False,
        eval_metric='logloss'
    ),
    'LightGBM': LGBMClassifier(     # CPU is default
        boosting_type='gbdt'
    ),
    'DecisionTree': DecisionTreeClassifier(),
    'Bagging': BaggingClassifier(
        estimator=DecisionTreeClassifier()
    )
}




def train_model(model, X_train, y_train):
    model.fit(X_train, y_train)
    return model

def evaluate_model(name, model, X_test, y_test):
    print(f"\n--- {name} Evaluation ---")
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    print(classification_report(y_test, y_pred))
    
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    if y_prob is not None:
        roc_auc = roc_auc_score(y_test, y_prob)
        print(f"ROC AUC: {roc_auc:.4f}")
        RocCurveDisplay.from_predictions(y_test, y_prob)
        plt.title(f"{name} ROC Curve")
        plt.show()

def plot_feature_importance(model, feature_names, name):
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        sorted_idx = importances.argsort()[::-1]
        top_n = min(10, len(importances))
        plt.figure(figsize=(8, 5))
        plt.barh(
            [feature_names[i] for i in sorted_idx[:top_n]][::-1],
            importances[sorted_idx[:top_n]][::-1]
        )
        plt.title(f"{name} Feature Importance (Top {top_n})")
        plt.xlabel("Importance Score")
        plt.tight_layout()
        plt.show()

trained_models = {}
for name, model in models.items():
    print(f"\nTraining {name}...")
    trained = train_model(model, X_train, y_train)
    trained_models[name] = trained
    evaluate_model(name, trained, X_test, y_test)
    plot_feature_importance(trained, X_train.columns, name)